# LLM Factor Mining Agent

QuantaAlpha 式闭环：多方向假设 → 因子生成（硬约束） → IS 网格搜索 → 横向评审 → OOS 验证 → 研究轨迹进化。

主循环：`one_batch()` — 一个 batch 完成一次完整的研究迭代。

In [1]:
import requests
import json
import os
import re
import sys

# Ensure local modules are discoverable
sys.path.insert(0, ".")

def load_env(path=".env"):
    """Load KEY=VALUE pairs from .env file (gitignored)."""
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())

load_env()

API_KEY = os.environ.get("OPENCODE_GO_API_KEY", "")
if not API_KEY:
    raise RuntimeError("Missing OPENCODE_GO_API_KEY: set it in .env or environment")
BASE_URL = "https://opencode.ai/zen/go/v1"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

MODEL = "deepseek-v4-pro"
print(f"Model: {MODEL}")

Model: deepseek-v4-pro


In [2]:
def chat(messages, model=None, max_tokens=8192, temperature=0.8):
    """Call OpenCode Go chat/completions API."""
    if model is None:
        model = MODEL

    data = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    resp = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, json=data, timeout=300)

    if resp.status_code != 200:
        raise RuntimeError(f"API error {resp.status_code}: {resp.text[:300]}")

    result = resp.json()
    msg = result["choices"][0]["message"]
    content = msg.get("content", "") or ""
    reasoning = msg.get("reasoning_content", "") or ""
    usage = result.get("usage", {})

    return {
        "content": content,
        "reasoning": reasoning,
        "answer": content if content.strip() else reasoning,
        "usage": usage,
        "model": result.get("model", ""),
    }


def quick_test():
    """Verify API connectivity."""
    resp = requests.get(f"{BASE_URL}/models", headers=HEADERS)
    if resp.status_code == 200:
        models = [m["id"] for m in resp.json().get("data", [])]
        print(f"[OK] {len(models)} models available: {models[:5]}...")
        result = chat([{"role": "user", "content": "Say hi."}], max_tokens=10)
        print(f"[OK] Tokens: in={result['usage'].get('prompt_tokens')}, out={result['usage'].get('completion_tokens')}")
    else:
        print(f"[FAIL] Models endpoint: {resp.status_code}")

quick_test()

[OK] 26 models available: ['minimax-m3', 'minimax-m2.7', 'minimax-m2.5', 'kimi-k3', 'kimi-k2.7-code']...
[OK] Tokens: in=86, out=10


## 初始化

回测引擎 + 研究轨迹（跨 batch 记忆）。

In [3]:
from backtest.engine import FactorBacktester
from agent.trajectory import ResearchTrajectory

# 先跑 BTCUSDT，后续扩展多币种
bt = FactorBacktester("data/BTCUSDT_1H.csv", commission_bps=6)
trajectory = ResearchTrajectory("trajectory.json")

print(f"Symbol: {bt.symbol}")
print(f"IS:     {bt.is_range} ({bt.is_rows} rows)")
print(f"OOS:    {bt.oos_range} ({bt.oos_rows} rows)")
print(f"Commission: {bt.commission_bps} bps")
print(f"轨迹方向数: {len(trajectory.data['directions'])}")

Symbol: BTCUSDT_1H
IS:     ('2020-01-01', '2025-06-29') (48145 rows)
OOS:    ('2025-06-29', '2026-06-29') (8760 rows)
Commission: 6 bps
轨迹方向数: 10


## Agent Batch 主循环

一个 batch：方向假设 → 因子生成 → IS网格搜索+代码筛选 → OOS → 轨迹更新。

筛选阈值在 `config.json`，改动后重新执行本 cell 生效。

In [4]:
from agent.hypothesis import generate_directions
from agent.factor_gen import generate_factor as gen_factor
from agent.judge import ask_oos_judge
from agent.config import load_config
from agent.screener import screen_factor

# 筛选阈值（改 config.json 即可调整）
cfg = load_config("config.json")


def one_batch(max_directions: int = 6) -> dict:
    """One full research iteration."""
    print("=" * 70)
    print("BATCH START")
    print(f"[筛选配置] {cfg}")

    # ---- Step 1: Hypothesis Agent ----
    print("\n[1] Hypothesis Agent: 生成研究方向...")
    hypo = generate_directions(chat, trajectory.summary())
    directions = hypo["directions"][:max_directions]
    if not directions:
        print("[FAIL] 方向解析失败")
        print(hypo["raw_response"][:500])
        return None
    for i, d in enumerate(directions):
        print(f"  [{i}] {d['name']} | {d['logic'][:60]}")

    # ---- Step 2: Factor Generation (with trajectory context) ----
    print("\n[2] Factor Generation...")
    factors = []
    for d in directions:
        ctx = trajectory.direction_context(d["name"])
        f = gen_factor(chat, d, trajectory_context=ctx)
        status = "OK" if f["valid"] else f"FAIL ({f['violation_reason'][:40]})"
        print(f"  [{d['name']}] {status}, retries={f['retries']}")
        factors.append(f)

    # ---- Step 3: IS Grid Search + Code Screening ----
    print("\n[3] IS Grid Search + Code Screening...")
    is_entries = {}
    for i, f in enumerate(factors):
        if not f["valid"]:
            continue
        try:
            result = bt.evaluate(f["formula"], f["category"])
        except Exception as e:
            print(f"  [{i}] {f['direction']['name']}: IS FAILED ({e})")
            continue
        isr = result["is_result"]

        # Sharpe 全负 -> 记录教训
        if isr["sharpe_max"] < 0:
            trajectory.add_attempt(
                f["direction"]["name"],
                f["formula"],
                isr,
                {"params": [], "pass_rate": "0/0"},
                "该方向IS Sharpe为负，方向本身可能错误",
            )
            trajectory.update_status(f["direction"]["name"], "failed")
            print(f"  [{i}] {f['direction']['name']}: Sharpe max={isr['sharpe_max']}<0 -> 记录教训")
            continue

        # 确定性筛选（替代LLM评审）
        screen = screen_factor(isr, cfg)
        print(f"  [{i}] {f['direction']['name']}: Sharpe max={isr['sharpe_max']}, "
              f"占比={isr['sharpe_positive_ratio']:.0%}, "
              f"入选{len(screen['selected_params'])}组, 通过={screen['passed']}")

        if not screen["passed"]:
            trajectory.add_attempt(
                f["direction"]["name"],
                f["formula"],
                isr,
                {"params": screen["selected_params"], "pass_rate": "0/0"},
                screen["reason"],
            )
            trajectory.update_status(f["direction"]["name"], "failed")
            print(f"    -> 记录教训: {screen['reason']}")
            continue

        stats = screen["selected_stats"]
        print(f"    -> 粗糙度={stats['roughness']['combined']}, "
              f"Sharpe范围={stats['sharpe_range']}")
        is_entries[i] = {
            "factor": f,
            "result": result,
            "selected_params": screen["selected_params"],
        }

    if not is_entries:
        print("[FAIL] 没有因子通过IS筛选")
        return None

    # ---- Step 4: OOS Test + trajectory update ----
    print(f"\n[4] OOS Test ({len(is_entries)} factors)...")
    oos_outcomes = []
    for i, entry in is_entries.items():
        f = entry["factor"]
        params = entry["selected_params"]
        is_result = entry["result"]
        try:
            oos = bt.evaluate_oos(f["formula"], params, is_result["is_result"])
        except Exception as e:
            print(f"  [{i}] {f['direction']['name']}: OOS FAILED ({e})")
            continue

        factor_info = f"{f['direction']['name']} | {f['formula'][:80]}"
        verdict = ask_oos_judge(chat, oos["oos_report"], factor_info)

        # OOS pass rate among selected params
        oos_lookup = {(r["window"], r["threshold"]): r
                      for r in oos["oos_result"]["results"]}
        pass_count = sum(1 for w, th in params
                         if oos_lookup.get((w, th), {}).get("sharpe", 0) > 0)

        # OOS 达标参数 (Sharpe >= oos_sharpe_min)，记录并保存
        qualified = [(w, th, oos_lookup[(w, th)]["sharpe"])
                     for w, th in params
                     if oos_lookup.get((w, th), {}).get("sharpe", 0)
                     >= cfg["oos_sharpe_min"]]
        print(f"  [{i}] OOS达标参数: {len(qualified)}/{len(params)}个 "
              f"(Sharpe >= {cfg['oos_sharpe_min']})")
        for w, th, s in qualified:
            print(f"      ({w}, {th}) Sharpe={s:.3f}")

        oos_summary = {"params": params,
                       "pass_rate": f"{pass_count}/{len(params)}",
                       "oos_qualified_params": [(w, th) for w, th, _ in qualified],
                       "oos_qualified_count": f"{len(qualified)}/{len(params)}"}

        # 学习记录: 0个达标时加提示
        learning = verdict["learning"]
        if len(qualified) == 0:
            learning = f"0/{len(params)}达标，OOS全面失效。{learning}"

        trajectory.add_attempt(
            f["direction"]["name"],
            f["formula"],
            is_result["is_result"],
            oos_summary,
            learning,
        )
        trajectory.update_status(
            f["direction"]["name"],
            "passed" if verdict["passed"] else "failed",
        )

        oos_outcomes.append({
            "fidx": i,
            "params": params,
            "oos": oos,
            "verdict": verdict,
        })

        print(f"  [{i}] {f['direction']['name']}: {verdict['verdict']}")
        print(f"    分析: {verdict['analysis'][:120]}")
        print(f"    教训: {verdict['learning'][:120]}")

    print("\n" + "=" * 70)
    print("BATCH DONE")
    return {
        "directions": directions,
        "factors": factors,
        "is_entries": is_entries,
        "oos_outcomes": oos_outcomes,
    }


print("one_batch() ready.")

one_batch() ready.


## 运行一个 Batch

In [5]:
batch = one_batch(max_directions=6)

BATCH START

[1] Hypothesis Agent: 生成研究方向...
  [0] 趋势效率过滤动量 | 以价格路径线性度/趋势效率作为开关，仅在趋势平滑且方向一致时持有，避免低效率震荡损耗。
  [1] 波动率压缩后突破 | 识别已实现波动率低位收敛后的区间突破方向，而非对波动率水平本身打分。
  [2] 量能确认条件动量 | 要求成交量累积方向与价格趋势一致才延续信号，不一致则过滤，避免单纯量价背离。
  [3] 历史高低点突破再延续 | 将近期高低点作为行为锚点，仅交易有效突破后的方向延续，而非锚定后的反转。
  [4] 收盘位置/日内买卖压力 | 用收盘价在当日高低价区间的相对位置连续累计，衡量买方/卖方尾盘控制力。
  [5] 隔夜跳空与日内修正 | 分解隔夜收益与日内收益，交易跳空过度后的日内反向修正或延续。

[2] Factor Generation...
  [趋势效率过滤动量] OK, retries=0
  [波动率压缩后突破] OK, retries=0
  [量能确认条件动量] OK, retries=1
  [历史高低点突破再延续] OK, retries=0
  [收盘位置/日内买卖压力] OK, retries=0
  [隔夜跳空与日内修正] OK, retries=0

[3] IS Grid Search...
  [0] 趋势效率过滤动量: Sharpe max=0.5853, rough=0.2048
  [1] 波动率压缩后突破: Sharpe max=-0.2399, rough=0.2773
    -> 记录教训: IS Sharpe全负
  [2] 量能确认条件动量: Sharpe max=-0.1023, rough=0.1933
    -> 记录教训: IS Sharpe全负
  [3] 历史高低点突破再延续: Sharpe max=0.4014, rough=0.2316
  [4] 收盘位置/日内买卖压力: Sharpe max=0.1715, rough=0.1506
  [5] 隔夜跳空与日内修正: Sharpe max=0.4877, rough=0.207

[4] IS Judge: 横向比较...
  选中 0 组: []
  [0] 趋势效率过滤动量: 被拒绝，记录教训
  

## 查看轨迹

In [7]:
# 查看研究轨迹（下个 batch 会喂给 Hypothesis Agent）
print(trajectory.summary())

方向「短期反转/过度反应」[failed] 尝试1次: 
  - IS Sharpe=-0.7069, 粗糙度=0.382, OOS={'params': [], 'pass_rate': '0/0'}
    教训: 该方向IS Sharpe为负，方向本身可能错误
方向「波动率聚集与波动率状态」[failed] 尝试1次: 
  - IS Sharpe=-0.0378, 粗糙度=0.1614, OOS={'params': [], 'pass_rate': '0/0'}
    教训: 该方向IS Sharpe为负，方向本身可能错误
方向「量价背离」[failed] 尝试1次: 
  - IS Sharpe=-0.3049, 粗糙度=0.2065, OOS={'params': [], 'pass_rate': '0/0'}
    教训: 该方向IS Sharpe为负，方向本身可能错误
方向「锚定效应」[failed] 尝试1次: 
  - IS Sharpe=-0.2341, 粗糙度=0.2999, OOS={'params': [], 'pass_rate': '0/0'}
    教训: 该方向IS Sharpe为负，方向本身可能错误
方向「时序动量/趋势延续」[failed] 尝试1次: 
  - IS Sharpe=0.6639, 粗糙度=0.185, OOS={'params': [], 'pass_rate': '0/0'}
    教训: IS表现差，被评审拒绝
方向「波动率偏度/尾部风险定价」[failed] 尝试1次: 
  - IS Sharpe=-0.2002, 粗糙度=0.1698, OOS={'params': [], 'pass_rate': '0/0'}
    教训: 该方向IS Sharpe为负，方向本身可能错误
方向「处置效应代理（盈亏敏感度）」[failed] 尝试1次: 
  - IS Sharpe=-0.6014, 粗糙度=0.1895, OOS={'params': [], 'pass_rate': '0/0'}
    教训: 该方向IS Sharpe为负，方向本身可能错误
方向「订单流不平衡/主动买卖压力」[failed] 尝试1次: 
  - IS Sharpe=-0.4641, 粗糙度=0.359, OOS=